# Transfer Learning

In [1]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import numpy as np
import torch
from sklearn.metrics import accuracy_score, f1_score
from src.load import PROCESSED_DIR, MODEL_DIR, make_loader, sample_labels
from src.models import Encoder, Classifier
from src.train_ssl import train_autoencoder
from src.train_cls import train_classifier

data = np.load(PROCESSED_DIR / 'splits.npz')

X_train = data['X_train']
X_val   = data['X_val']
X_test  = data['X_test']
y_train = data['y_train']
y_val   = data['y_val']
y_test  = data['y_test']

train_loader = make_loader(X_train, y_train, batch_size=128, shuffle=True)
val_loader   = make_loader(X_val, y_val, batch_size=128)
test_loader  = make_loader(X_test, y_test, batch_size=128)

train_loader_ssl = make_loader(X_train, batch_size=128, shuffle=True)

In [2]:
encoder = Encoder()
encoder.load_state_dict(torch.load(MODEL_DIR / "encoder_ae.pth"))

<All keys matched successfully>

In [9]:
device = "cpu"
encoder = Encoder()
encoder.load_state_dict(torch.load(MODEL_DIR / "encoder_ae.pth"))

model = train_classifier(
    encoder,
    train_loader,
    val_loader,
    epochs=10,
    device=device
)

torch.save(model.state_dict(), MODEL_DIR / "ae_full.pth")

Epoch 1 | Loss: 1.5650 | Val Acc: 0.5494
Epoch 2 | Loss: 0.8445 | Val Acc: 0.6680
Epoch 3 | Loss: 0.6627 | Val Acc: 0.6795
Epoch 4 | Loss: 0.5705 | Val Acc: 0.7523
Epoch 5 | Loss: 0.5126 | Val Acc: 0.8029
Epoch 6 | Loss: 0.4668 | Val Acc: 0.8034
Epoch 7 | Loss: 0.4257 | Val Acc: 0.7908
Epoch 8 | Loss: 0.3841 | Val Acc: 0.7489
Epoch 9 | Loss: 0.3655 | Val Acc: 0.8140
Epoch 10 | Loss: 0.3518 | Val Acc: 0.8222


## Evaluate - Full data

In [3]:
def evaluate(model, loader, device="cpu"):
    model.eval()

    all_preds, all_labels = [], []

    with torch.no_grad():
        for X, y in loader:
            X = X.to(device)
            out = model(X)
            preds = out.argmax(dim=1).cpu().numpy()

            all_preds.extend(preds)
            all_labels.extend(y.numpy())

    acc = accuracy_score(all_labels, all_preds)
    f1  = f1_score(all_labels, all_preds, average='macro')

    return acc, f1

In [10]:
encoder = Encoder()
model = Classifier(encoder).to("cpu")

model.load_state_dict(torch.load(MODEL_DIR / "ae_full.pth"))

test_acc, test_f1 = evaluate(model, test_loader)

print("AE Full -> Test Acc:", test_acc)
print("AE Full -> Test F1 :", test_f1)

AE Full -> Test Acc: 0.8262650602409638
AE Full -> Test F1 : 0.8275866328919128


## MAE

In [4]:
device = "cpu"
encoder = Encoder()
encoder.load_state_dict(torch.load(MODEL_DIR / "encoder_mae.pth"))

model = train_classifier(
    encoder,
    train_loader,
    val_loader,
    epochs=10,
    device=device
)

torch.save(model.state_dict(), MODEL_DIR / "mae_full.pth")

Epoch 1 | Loss: 1.4028 | Val Acc: 0.7022
Epoch 2 | Loss: 0.7416 | Val Acc: 0.7976
Epoch 3 | Loss: 0.5630 | Val Acc: 0.8357
Epoch 4 | Loss: 0.4634 | Val Acc: 0.8834
Epoch 5 | Loss: 0.3967 | Val Acc: 0.8713
Epoch 6 | Loss: 0.3427 | Val Acc: 0.8867
Epoch 7 | Loss: 0.3259 | Val Acc: 0.9186
Epoch 8 | Loss: 0.2646 | Val Acc: 0.9195
Epoch 9 | Loss: 0.2620 | Val Acc: 0.8911
Epoch 10 | Loss: 0.2464 | Val Acc: 0.9316


In [5]:
encoder = Encoder()
model = Classifier(encoder).to("cpu")

model.load_state_dict(torch.load(MODEL_DIR / "mae_full.pth"))

test_acc, test_f1 = evaluate(model, test_loader)

print("MAE Full -> Test Acc:", test_acc)
print("MAE Full -> Test F1 :", test_f1)

MAE Full -> Test Acc: 0.9368674698795181
MAE Full -> Test F1 : 0.9464720619780594


## Sample 20%

In [6]:
X_small, y_small = sample_labels(X_train, y_train, fraction=0.2)    
train_loader_small = make_loader(X_small, y_small, batch_size=128, shuffle=True)

In [7]:
encoder = Encoder()
encoder.load_state_dict(torch.load(MODEL_DIR / "encoder_mae.pth"))

model_small = train_classifier(
    encoder,
    train_loader_small,
    val_loader,
    epochs=15,
    device=device
)

torch.save(model_small.state_dict(), MODEL_DIR / "mae_20.pth")

Epoch 1 | Loss: 2.2039 | Val Acc: 0.4612
Epoch 2 | Loss: 1.5008 | Val Acc: 0.6246
Epoch 3 | Loss: 1.1919 | Val Acc: 0.6294
Epoch 4 | Loss: 1.0056 | Val Acc: 0.7277
Epoch 5 | Loss: 0.8805 | Val Acc: 0.7195
Epoch 6 | Loss: 0.7865 | Val Acc: 0.7701
Epoch 7 | Loss: 0.6703 | Val Acc: 0.7561
Epoch 8 | Loss: 0.6063 | Val Acc: 0.7802
Epoch 9 | Loss: 0.5541 | Val Acc: 0.8178
Epoch 10 | Loss: 0.5228 | Val Acc: 0.7807
Epoch 11 | Loss: 0.4726 | Val Acc: 0.8029
Epoch 12 | Loss: 0.4340 | Val Acc: 0.8357
Epoch 13 | Loss: 0.4160 | Val Acc: 0.8323
Epoch 14 | Loss: 0.3905 | Val Acc: 0.8328
Epoch 15 | Loss: 0.3664 | Val Acc: 0.8120


In [8]:
encoder = Encoder()
model_small = Classifier(encoder).to(device)

model_small.load_state_dict(torch.load(MODEL_DIR / "mae_20.pth"))

test_acc, test_f1 = evaluate(model_small, test_loader)

print("MAE 20% → Test Acc:", test_acc)
print("MAE 20% → Test F1 :", test_f1)

MAE 20% → Test Acc: 0.8356626506024096
MAE 20% → Test F1 : 0.8268138063600321
